In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from pathlib import Path

from src.scenarios.get_metrics_by_freq import run_script

from src.data_utils import get_sample_from_row_original
from transformers import BartTokenizer, BartForConditionalGeneration
from src.inference_utils import GenerativeModelWithCaching, GenerativeModel

In [2]:
CURDIR = Path.cwd()

DATADIR = CURDIR / "data"
assert DATADIR.exists()

MODELS_DIR = CURDIR / "models"
assert MODELS_DIR.exists()

MODEL_ID = MODELS_DIR / "BART_base_tune"
assert MODEL_ID.exists()

In [3]:
tokenizer = BartTokenizer.from_pretrained(MODEL_ID)

In [4]:
model_ = BartForConditionalGeneration.from_pretrained(MODEL_ID)

In [5]:
model_.num_parameters()

139420416

In [6]:
model__ = GenerativeModel(model_, tokenizer, verbose=False)

In [7]:
model = GenerativeModelWithCaching(model__)

In [8]:
df = pd.read_csv(DATADIR / "original/test.csv", sep="\t", index_col=0)

In [9]:
df.head()

,form,pos,feats,lemma,freq_rank,subset,filename,split
0,Ещё,ADV,Degree:Pos,еще,45,rubic_data-master,ru_taiga-ud-test.conllu,holdout
1,зимой,NOUN,Animacy:Inan Case:Ins Gender:Fem Number:Sing,зима,1147,rubic_data-master,ru_taiga-ud-test.conllu,holdout
2,в,ADP,NaN,в,2,rubic_data-master,ru_taiga-ud-test.conllu,holdout
3,армиях,NOUN,Animacy:Inan Case:Loc Gender:Fem Number:Plur,армия,602,rubic_data-master,ru_taiga-ud-test.conllu,holdout
4,ДНР,PROPN,Abbr:Yes NameType:Geo,ДНР,other,rubic_data-master,ru_taiga-ud-test.conllu,holdout


In [10]:
df.shape

(153991, 8)

In [11]:
df["sample"] = df.apply(lambda row: get_sample_from_row_original(row)[0], axis=1)
uniq_samples = df["sample"].unique().tolist()
print(len(uniq_samples))
preds = model.predict(uniq_samples)

38963


609it [02:46,  3.66it/s]                         


In [12]:
all_, holdout_, unknown_ = run_script(df, model.predict)

     form    pos                                         feats  lemma  \
0     Ещё    ADV                                    Degree:Pos    еще   
1   зимой   NOUN  Animacy:Inan Case:Ins Gender:Fem Number:Sing   зима   
2       в    ADP                                           NaN      в   
3  армиях   NOUN  Animacy:Inan Case:Loc Gender:Fem Number:Plur  армия   
4     ДНР  PROPN                         Abbr:Yes NameType:Geo    ДНР   

  freq_rank             subset                 filename    split  \
0        45  rubic_data-master  ru_taiga-ud-test.conllu  holdout   
1      1147  rubic_data-master  ru_taiga-ud-test.conllu  holdout   
2         2  rubic_data-master  ru_taiga-ud-test.conllu  holdout   
3       602  rubic_data-master  ru_taiga-ud-test.conllu  holdout   
4     other  rubic_data-master  ru_taiga-ud-test.conllu  holdout   

                                              sample  
0                                 Ещё ADV Degree:Pos  
1  зимой NOUN Animacy:Inan Case:Ins Gender

2397it [00:00, 69701.03it/s]            
2164it [00:00, 73799.67it/s]            
233it [00:00, 39760.48it/s]            


In [13]:
all_

,class,lAcc,lAcc (norm),CER (total),CER (errors)
0,1-100,0.999,0.999,0.001,0.490
1,101-1000,0.992,0.993,0.003,0.328
2,1001-10000,0.982,0.985,0.004,0.228
3,10001-n,0.984,0.988,0.004,0.216
4,all,0.990,0.992,0.003,0.245


In [14]:
holdout_

,class,lAcc,lAcc (norm),CER (total),CER (errors)
0,1-100,0.999,0.999,0.000,0.509
1,101-1000,0.995,0.995,0.002,0.340
2,1001-10000,0.989,0.991,0.003,0.262
3,10001-n,0.996,0.997,0.001,0.195
4,all,0.996,0.996,0.001,0.274


In [15]:
unknown_

,class,lAcc,lAcc (norm),CER (total),CER (errors)
0,1-100,0.849,0.856,0.071,0.454
1,101-1000,0.921,0.927,0.024,0.308
2,1001-10000,0.946,0.952,0.010,0.192
3,10001-n,0.934,0.949,0.016,0.222
4,all,0.936,0.948,0.015,0.227


In [16]:
df = pd.read_csv(DATADIR / "additional/poetic_sample_18.csv", sep="\t", index_col=0)
df["sample"] = df.apply(lambda row: get_sample_from_row_original(row)[0], axis=1)
uniq_samples = df["sample"].unique().tolist()
print(len(uniq_samples))
preds = model.predict(uniq_samples)

136314


2130it [08:07,  4.37it/s]                          


In [17]:
all_, holdout_, unknown_ = run_script(df, model.predict)

            form   pos                                              feats  \
0          среду  NOUN       Animacy:Inan Case:Acc Gender:Fem Number:Sing   
1          Эолов   ADJ        Case:Nom Degree:Pos Gender:Masc Number:Sing   
2      Умножайся  VERB  Aspect:Imp Mood:Imp Number:Sing Person:2 Trans...   
3          ощутя  VERB  Aspect:Perf Tense:Past Transit:Tran VerbForm:C...   
4  руководствует  VERB  Aspect:Imp Mood:Ind Number:Sing Person:3 Tense...   

             lemma                 filename freq_rank    split  \
0            среда  poetic_sample_18.conllu       836  holdout   
1            эолов  poetic_sample_18.conllu     other  unknown   
2       умножаться  poetic_sample_18.conllu     other  unknown   
3          ощутить  poetic_sample_18.conllu      3063  unknown   
4  руководствовать  poetic_sample_18.conllu     other  unknown   

                                              sample  
0  среду NOUN Animacy:Inan Case:Acc Gender:Fem Nu...  
1  Эолов ADJ Case:Nom Degree:P

2124it [00:00, 45957.15it/s]            
743it [00:00, 36681.28it/s]            
1382it [00:00, 38245.51it/s]            


In [18]:
all_

,class,lAcc,lAcc (norm),CER (total),CER (errors)
0,1-100,0.913,0.920,0.058,0.653
1,101-1000,0.954,0.958,0.017,0.355
2,1001-10000,0.956,0.961,0.012,0.275
3,10001-n,0.913,0.932,0.020,0.204
4,all,0.933,0.945,0.017,0.241


In [19]:
holdout_

,class,lAcc,lAcc (norm),CER (total),CER (errors)
0,1-100,0.967,0.967,0.019,0.523
1,101-1000,0.985,0.986,0.005,0.342
2,1001-10000,0.985,0.988,0.004,0.250
3,10001-n,0.968,0.978,0.007,0.214
4,all,0.980,0.984,0.005,0.268


In [20]:
unknown_

,class,lAcc,lAcc (norm),CER (total),CER (errors)
0,1-100,0.807,0.825,0.138,0.700
1,101-1000,0.895,0.905,0.038,0.359
2,1001-10000,0.925,0.933,0.021,0.280
3,10001-n,0.903,0.923,0.022,0.204
4,all,0.907,0.923,0.024,0.238


In [21]:
df = pd.read_csv(DATADIR / "additional/poetic_sample_19.csv", sep="\t", index_col=0)
df["sample"] = df.apply(lambda row: get_sample_from_row_original(row)[0], axis=1)
uniq_samples = df["sample"].unique().tolist()
print(len(uniq_samples))
preds = model.predict(uniq_samples)

331521


5181it [19:30,  4.43it/s]                          


In [22]:
all_, holdout_, unknown_ = run_script(df, model.predict)

          form    pos                                              feats  \
0     растаяли   VERB  Aspect:Perf Mood:Ind Number:Plur Tense:Past Tr...   
1      Цезарем  PROPN  Animacy:Anim Case:Ins Gender:Masc NameType:Giv...   
2     отЕерсто   VERB  Aspect:Perf Case:Nom Gender:Neut Number:Sing T...   
3  неотступную    ADJ         Case:Acc Degree:Pos Gender:Fem Number:Sing   
4  скорбно-тих    ADJ   Degree:Pos Gender:Masc Number:Sing Variant:Short   

           lemma                 filename freq_rank    split  \
0       растаять  poetic_sample_19.conllu      8753  holdout   
1         Цезарь  poetic_sample_19.conllu     other  unknown   
2     отверзнуть  poetic_sample_19.conllu     other  unknown   
3    неотступный  poetic_sample_19.conllu     other  unknown   
4  скорбно-тихий  poetic_sample_19.conllu     other  unknown   

                                              sample  
0  растаяли VERB Aspect:Perf Mood:Ind Number:Plur...  
1  Цезарем PROPN Animacy:Anim Case:Ins Gender:Ma

5182it [00:00, 26830.21it/s]                          
1486it [00:00, 27790.97it/s]            
3696it [00:00, 30146.68it/s]                          


In [23]:
all_

,class,lAcc,lAcc (norm),CER (total),CER (errors)
0,1-100,0.909,0.919,0.059,0.606
1,101-1000,0.964,0.969,0.014,0.371
2,1001-10000,0.965,0.969,0.009,0.250
3,10001-n,0.927,0.945,0.016,0.191
4,all,0.942,0.954,0.014,0.215


In [24]:
holdout_

,class,lAcc,lAcc (norm),CER (total),CER (errors)
0,1-100,0.962,0.962,0.023,0.533
1,101-1000,0.986,0.988,0.005,0.342
2,1001-10000,0.984,0.987,0.004,0.238
3,10001-n,0.967,0.976,0.007,0.211
4,all,0.978,0.983,0.005,0.248


In [25]:
unknown_

,class,lAcc,lAcc (norm),CER (total),CER (errors)
0,1-100,0.820,0.846,0.121,0.633
1,101-1000,0.928,0.938,0.028,0.380
2,1001-10000,0.949,0.954,0.013,0.253
3,10001-n,0.920,0.939,0.018,0.190
4,all,0.927,0.942,0.018,0.211


In [26]:
df = pd.read_csv(DATADIR / "additional/school.csv", sep="\t", index_col=0)
df["sample"] = df.apply(lambda row: get_sample_from_row_original(row)[0], axis=1)
uniq_samples = df["sample"].unique().tolist()
print(len(uniq_samples))
preds = model.predict(uniq_samples)

183253


2864it [09:14,  5.16it/s]                          


In [27]:
all_, holdout_, unknown_ = run_script(df, model.predict)

      form   pos                                              feats    lemma  \
0  История  NOUN       Animacy:Inan Case:Nom Gender:Fem Number:Sing  история   
1    языка  NOUN      Animacy:Inan Case:Gen Gender:Masc Number:Sing     язык   
2  История  NOUN       Animacy:Inan Case:Nom Gender:Fem Number:Sing  история   
3    языка  NOUN      Animacy:Inan Case:Gen Gender:Masc Number:Sing     язык   
4  изучает  VERB  Aspect:Imp Mood:Ind Number:Sing Person:3 Tense...  изучать   

                  subset         filename    split freq_rank  \
0  uch-nauch/encicl_phil  encicl66.conllu  holdout       214   
1  uch-nauch/encicl_phil  encicl66.conllu  holdout       306   
2  uch-nauch/encicl_phil  encicl66.conllu  holdout       214   
3  uch-nauch/encicl_phil  encicl66.conllu  holdout       306   
4  uch-nauch/encicl_phil  encicl66.conllu  holdout      2084   

                                              sample  
0  История NOUN Animacy:Inan Case:Nom Gender:Fem ...  
1  языка NOUN Animacy:In

25024it [00:00, 39612.32it/s]                           
12900it [00:00, 40556.94it/s]                           
12125it [00:00, 74055.64it/s]                          


In [28]:
all_

,class,lAcc,lAcc (norm),CER (total),CER (errors)
0,1-100,0.998,0.998,0.001,0.506
1,101-1000,0.991,0.993,0.003,0.338
2,1001-10000,0.984,0.988,0.004,0.230
3,10001-n,0.986,0.990,0.003,0.218
4,all,0.990,0.992,0.003,0.259


In [29]:
holdout_

,class,lAcc,lAcc (norm),CER (total),CER (errors)
0,1-100,0.999,0.999,0.001,0.443
1,101-1000,0.990,0.993,0.003,0.298
2,1001-10000,0.986,0.990,0.003,0.220
3,10001-n,0.977,0.985,0.004,0.188
4,all,0.990,0.993,0.002,0.244


In [30]:
unknown_

,class,lAcc,lAcc (norm),CER (total),CER (errors)
0,1-100,0.997,0.997,0.002,0.531
1,101-1000,0.994,0.994,0.006,0.962
2,1001-10000,0.977,0.981,0.006,0.263
3,10001-n,0.987,0.990,0.003,0.227
4,all,0.990,0.992,0.003,0.275


In [31]:
def print_cfg(model):
    print("Model configuration:")
    print(f"Encoder layers: {model.config.encoder_layers}")
    print(f"Decoder layers: {model.config.decoder_layers}")
    print(f"Encoder attention heads: {model.config.encoder_attention_heads}")
    print(f"Decoder attention heads: {model.config.decoder_attention_heads}")
    print(f"Hidden size: {model.config.d_model}")
    print(f"FFN dimension: {model.config.encoder_ffn_dim} / {model.config.decoder_ffn_dim}")
    print(f"Total parameters: {model.num_parameters():,}")

In [32]:
print_cfg(model_)

Model configuration:
Encoder layers: 6
Decoder layers: 6
Encoder attention heads: 12
Decoder attention heads: 12
Hidden size: 768
FFN dimension: 3072 / 3072
Total parameters: 139,420,416
